In [2]:
import pandas as pd

from pathlib import Path
from sklearn.metrics import precision_score, recall_score


# ---------------------------------------------------------
# SENSOR COMBINATION ANALYSIS - TASK 1.1
# ---------------------------------------------------------

SIGNALS = [
    "accelerometer",
    "gravity",
    "gyros",
    "lin_accel",
    "game_rot_vec",
    "magn_field"
]

MOVEMENT_NAMES = {
    0: "Absence of movement",
    1: "Upward Block",
    2: "Hammering Inward Block",
    3: "Extended Outward Block",
    4: "Outward Downward Block",
    5: "Rear Elbow Block"
}


# ---------------------------------------------------------
# LOAD PREDICTIONS
# ---------------------------------------------------------

predictions_df = pd.read_csv("../experimental_results/task_1_1/task_1_1_predictions.csv")


# ---------------------------------------------------------
# COMPUTE PRECISION AND RECALL FROM PREDICTIONS
# ---------------------------------------------------------

results = []

for experiment in predictions_df["experiment"].unique():

    experiment_df = predictions_df[predictions_df["experiment"] == experiment]

    y_true = experiment_df["true_label"]
    y_pred = experiment_df["predicted_label"]

    for movement in sorted(y_true.unique()):

        y_true_binary = (y_true == movement).astype(int)
        y_pred_binary = (y_pred == movement).astype(int)

        results.append({
            "experiment": experiment,
            "movement": movement,
            "precision": precision_score(y_true_binary, y_pred_binary, zero_division=0),
            "recall": recall_score(y_true_binary, y_pred_binary, zero_division=0)
        })


results_df = pd.DataFrame(results)


# ---------------------------------------------------------
# KEEP ONLY SENSOR ABLATIONS
# ---------------------------------------------------------

# Exclude axis experiments such as remove_x, remove_y, ...
# Keep baseline and experiments involving sensor signals.

signal_mask = (
    (results_df["experiment"] == "baseline") |
    results_df["experiment"].str.contains("|".join(SIGNALS), regex=True)
)

signal_results = results_df[signal_mask].copy()


# ---------------------------------------------------------
# COUNT REMOVED SENSORS
# ---------------------------------------------------------

def count_removed_signals(experiment):

    if experiment == "baseline":
        return 0

    return sum(signal in experiment for signal in SIGNALS)


signal_results["n_removed_signals"] = signal_results["experiment"].apply(count_removed_signals)


# ---------------------------------------------------------
# CHECK EXPERIMENTS
# ---------------------------------------------------------

experiments_check = (
    signal_results[["experiment", "n_removed_signals"]]
    .drop_duplicates()
    .sort_values(["n_removed_signals", "experiment"])
)

print("\n=== EXPERIMENTS ===")
print(experiments_check.to_string(index=False))


# ---------------------------------------------------------
# BASELINE
# ---------------------------------------------------------

baseline = (
    signal_results[signal_results["experiment"] == "baseline"]
    .set_index("movement")[["precision", "recall"]]
)

print("\n=== BASELINE ===")
print(baseline.rename(index=MOVEMENT_NAMES))


# ---------------------------------------------------------
# DELTA PRECISION AND RECALL
# ---------------------------------------------------------

signal_results["delta_precision"] = signal_results.apply(
    lambda row: row["precision"] - baseline.loc[row["movement"], "precision"],
    axis=1
)

signal_results["delta_recall"] = signal_results.apply(
    lambda row: row["recall"] - baseline.loc[row["movement"], "recall"],
    axis=1
)


# ---------------------------------------------------------
# KEEP ABLATION EXPERIMENTS
# ---------------------------------------------------------

combinations = signal_results[signal_results["experiment"] != "baseline"].copy()


# ---------------------------------------------------------
# 1. SUMMARY BY NUMBER OF REMOVED SENSORS
# ---------------------------------------------------------

summary_by_n = (
    combinations
    .groupby("n_removed_signals")
    .agg(
        mean_delta_precision=("delta_precision", "mean"),
        mean_delta_recall=("delta_recall", "mean"),
        min_delta_precision=("delta_precision", "min"),
        min_delta_recall=("delta_recall", "min")
    )
    .round(3)
)

print("\n=== EFFECT OF NUMBER OF REMOVED SENSORS ===")
print(summary_by_n)


# ---------------------------------------------------------
# 2. SUMMARY BY MOVEMENT
# ---------------------------------------------------------

movement_summary = (
    combinations
    .groupby("movement")
    .agg(
        mean_delta_precision=("delta_precision", "mean"),
        mean_delta_recall=("delta_recall", "mean"),
        worst_delta_precision=("delta_precision", "min"),
        worst_delta_recall=("delta_recall", "min")
    )
    .round(3)
)

movement_summary.index = movement_summary.index.map(MOVEMENT_NAMES)

print("\n=== EFFECT BY MOVEMENT ===")
print(movement_summary)


# ---------------------------------------------------------
# 3. SUMMARY BY EXPERIMENT
# ---------------------------------------------------------

experiment_summary = (
    combinations
    .groupby(["experiment", "n_removed_signals"])
    .agg(
        mean_delta_precision=("delta_precision", "mean"),
        mean_delta_recall=("delta_recall", "mean"),
        worst_delta_precision=("delta_precision", "min"),
        worst_delta_recall=("delta_recall", "min")
    )
    .reset_index()
    .round(3)
)


# ---------------------------------------------------------
# WORST SENSOR COMBINATIONS
# ---------------------------------------------------------

worst_combinations = (
    experiment_summary
    .sort_values(["mean_delta_recall", "mean_delta_precision"])
    .head(15)
)

print("\n=== WORST SENSOR COMBINATIONS ===")
print(worst_combinations.to_string(index=False))


# ---------------------------------------------------------
# MOST ROBUST SENSOR COMBINATIONS
# ---------------------------------------------------------

best_combinations = (
    experiment_summary
    .sort_values(["mean_delta_recall", "mean_delta_precision"], ascending=False)
    .head(15)
)

print("\n=== MOST ROBUST SENSOR COMBINATIONS ===")
print(best_combinations.to_string(index=False))


# ---------------------------------------------------------
# 4. RECALL DELTA TABLE
# ---------------------------------------------------------

recall_delta_table = signal_results.pivot(
    index="experiment",
    columns="movement",
    values="delta_recall"
)

recall_delta_table = recall_delta_table.rename(columns=MOVEMENT_NAMES)

recall_delta_table["n_removed_signals"] = recall_delta_table.index.map(count_removed_signals)

recall_delta_table = recall_delta_table.sort_values("n_removed_signals").round(3)

print("\n=== RECALL DELTA TABLE ===")
print(recall_delta_table)


# ---------------------------------------------------------
# 5. PRECISION DELTA TABLE
# ---------------------------------------------------------

precision_delta_table = signal_results.pivot(
    index="experiment",
    columns="movement",
    values="delta_precision"
)

precision_delta_table = precision_delta_table.rename(columns=MOVEMENT_NAMES)

precision_delta_table["n_removed_signals"] = precision_delta_table.index.map(count_removed_signals)

precision_delta_table = precision_delta_table.sort_values("n_removed_signals").round(3)

print("\n=== PRECISION DELTA TABLE ===")
print(precision_delta_table)


# ---------------------------------------------------------
# SAVE RESULTS
# ---------------------------------------------------------

output_dir = Path("../experimental_results/task_1_1/signal_combinations")
output_dir.mkdir(parents=True, exist_ok=True)

summary_by_n.to_csv(output_dir / "sensor_combinations_by_number.csv")

movement_summary.to_csv(output_dir / "sensor_combinations_by_movement.csv")

experiment_summary.to_csv(
    output_dir / "sensor_combinations_summary.csv",
    index=False
)

recall_delta_table.to_csv(
    output_dir / "sensor_combinations_recall_delta.csv"
)

precision_delta_table.to_csv(
    output_dir / "sensor_combinations_precision_delta.csv"
)


=== EXPERIMENTS ===
                                                    experiment  n_removed_signals
                                                      baseline                  0
                                          remove_accelerometer                  1
                                           remove_game_rot_vec                  1
                                                remove_gravity                  1
                                                  remove_gyros                  1
                                              remove_lin_accel                  1
                                             remove_magn_field                  1
                             remove_accelerometer_game_rot_vec                  2
                                  remove_accelerometer_gravity                  2
                                    remove_accelerometer_gyros                  2
                                remove_accelerometer_lin_accel               